# WtDtCore 架构

## 概述

WtDtCore（数据传输核心）是WonderTrader框架中负责行情数据接收、处理、存储和分发的核心模块。该模块采用分层架构设计，实现了高性能、高可靠性的实时数据处理系统。
```mermaid
graph TB
    subgraph "第1层：接口定义层"
        IDataCaster[IDataCaster.h<br/>广播器接口]
    end
    
    subgraph "第2层：核心管理层"
        DataManager[DataManager.h/.cpp<br/>数据管理中枢]
        StateMonitor[StateMonitor.h/.cpp<br/>状态监控器]
        ParserAdapter[ParserAdapter.h/.cpp<br/>解析器适配器]
    end
    
    subgraph "第3层：功能实现层"
        UDPCaster[UDPCaster.h/.cpp<br/>UDP广播实现]
        ShmCaster[ShmCaster.h/.cpp<br/>共享内存广播实现]
        IndexFactory[IndexFactory.h/.cpp<br/>指数工厂]
        IndexWorker[IndexWorker.h/.cpp<br/>指数计算器]
    end
    
    subgraph "第4层：辅助工具层"
        WtHelper[WtHelper.h/.cpp<br/>路径工具]
        StatHelper[StatHelper.hpp<br/>统计工具]
    end
    
    %% 依赖关系
    DataManager --> IDataCaster
    UDPCaster --> IDataCaster
    ShmCaster --> IDataCaster
    ParserAdapter --> DataManager
    StateMonitor --> DataManager
    IndexFactory --> DataManager
    IndexWorker --> IndexFactory
    DataManager --> WtHelper
    UDPCaster --> StatHelper
    ShmCaster --> StatHelper
```

## 核心文件详细分析

### 1. 数据管理中枢 - DataManager

```mermaid
graph LR
    subgraph "DataManager 核心职责"
        A[行情数据接收] --> B[数据验证与过滤]
        B --> C[存储引擎调度]
        C --> D[广播器管理]
        D --> E[状态控制集成]
    end
    
    subgraph "接口实现"
        F[IDataWriterSink<br/>为Writer提供回调]
    end
    
    subgraph "设计模式"
        G[外观模式<br/>Facade Pattern]
        H[适配器模式<br/>Adapter Pattern]
        I[策略模式<br/>Strategy Pattern]
    end
    
    A -.-> F
    C -.-> G
    F -.-> H
    D -.-> I
```

**核心特点：**
- **中枢角色**：作为数据流转的核心枢纽，协调Parser、Writer、Caster、StateMonitor
- **双重身份**：既是管理门面，又是Writer的回调接收器
- **动态加载**：支持运行时加载不同的存储引擎（WtDataStorage、自定义Writer）
- **多播支持**：同时支持多个数据广播器并行工作

### 2. 行情解析适配器 - ParserAdapter

```mermaid
graph TB
    subgraph "ParserAdapter 工作流程"
        A[动态加载Parser模块] --> B[解析过滤器配置]
        B --> C[智能订阅策略]
        C --> D[行情数据接收]
        D --> E[数据验证与转发]
    end
    
    subgraph "订阅策略优先级"
        F[1. Code Filter<br/>合约代码过滤]
        G[2. Exchange Filter<br/>交易所过滤]
        H[3. Full Market<br/>全市场订阅]
        F --> G --> H
    end
    
    subgraph "适配器模式应用"
        I[IParserApi<br/>行情解析器接口]
        J[IParserSpi<br/>行情回调接口]
        K[ParserAdapter<br/>适配器实现]
        I --> K
        K --> J
    end
```

**核心特点：**
- **适配器模式**：将不同厂商的Parser统一适配到框架接口
- **智能订阅**：支持品种级、交易所级、全市场级订阅策略
- **动态加载**：运行时加载Parser动态库，支持多种行情源
- **数据过滤**：在接入层就进行数据过滤，提高处理效率

### 3. 状态监控器 - StateMonitor

```mermaid
stateDiagram-v2
    [*] --> SS_ORIGINAL : 系统启动
    SS_ORIGINAL --> SS_INITIALIZED : 到达初始化时间
    SS_INITIALIZED --> SS_RECEIVING : 到达开盘时间
    SS_RECEIVING --> SS_PAUSED : 中途休盘
    SS_PAUSED --> SS_RECEIVING : 恢复交易
    SS_RECEIVING --> SS_CLOSED : 到达收盘时间
    SS_CLOSED --> SS_PROCING : 到达盘后处理时间
    SS_PROCING --> SS_PROCED : 处理完成
    SS_PROCED --> SS_ORIGINAL : 下一交易日
    
    SS_ORIGINAL --> SS_Holiday : 检测到节假日
    SS_INITIALIZED --> SS_Holiday : 检测到节假日
    SS_Holiday --> SS_ORIGINAL : 下一交易日
    
    note right of SS_RECEIVING : 可接收数据状态
    note right of SS_PROCING : 触发历史数据转储
```

**核心特点：**
- **有限状态机**：精确控制数据接收和处理的时机
- **多时段管理**：支持不同交易时段的独立状态管理
- **自动转换**：基于时间和交易日历自动进行状态转换
- **盘后处理**：自动触发历史数据转储和缓存清理

### 4. 数据广播器架构

```mermaid
graph TB
    subgraph "IDataCaster 接口层"
        A[IDataCaster<br/>统一广播接口]
    end
    
    subgraph "具体实现层"
        B[UDPCaster<br/>UDP网络广播]
        C[ShmCaster<br/>共享内存广播]
        D[CustomCaster<br/>自定义广播器]
    end
    
    subgraph "UDPCaster 特性"
        E[单播模式<br/>点对点传输]
        F[广播模式<br/>局域网广播]
        G[组播模式<br/>组播传输]
        H[订阅服务<br/>客户端订阅]
    end
    
    subgraph "ShmCaster 特性"
        I[无锁环形队列<br/>Lock-Free Ring Buffer]
        J[纳秒级延迟<br/>极致性能]
        K[进程间通信<br/>本地IPC]
        L[联合体设计<br/>节省内存]
    end
    
    A --> B
    A --> C
    A --> D
    
    B --> E
    B --> F
    B --> G
    B --> H
    
    C --> I
    C --> J
    C --> K
    C --> L
```

**性能对比：**
- **UDPCaster**：延迟1-10ms，适用于网络分发，支持跨机器
- **ShmCaster**：延迟<100ns，适用于本地进程间通信，性能极致

### 5. 指数计算模块

```mermaid
graph TB
    subgraph "IndexFactory 工厂管理"
        A[指数配置加载] --> B[Worker创建管理]
        B --> C[成分合约订阅]
        C --> D[行情数据分发]
        D --> E[指数结果收集]
    end
    
    subgraph "IndexWorker 计算引擎"
        F[权重算法选择]
        G[触发策略配置]
        H[实时指数计算]
        I[延时计算优化]
    end
    
    subgraph "权重算法类型"
        J[算法0: 固定权重<br/>Fixed Weight]
        K[算法1: 动态总持权重<br/>Dynamic Interest]
        L[算法2: 动态成交量权重<br/>Dynamic Volume]
    end
    
    F --> J
    F --> K
    F --> L
    
    A --> F
    D --> G
    G --> H
    H --> I
```

**计算公式：**
- **固定权重**：`指数 = Σ(价格 × 权重) / 总权重 × 标准化系数`
- **动态总持**：`指数 = Σ(价格 × 持仓量 × 权重) / Σ持仓量 / 总权重 × 标准化系数`
- **动态成交量**：`指数 = Σ(价格 × 成交量 × 权重) / Σ成交量 / 总权重 × 标准化系数`

# --------功能实现层--------

# 指数计算器 IndexWorker.h/cpp

# --------核心管理层--------

# 交易时段状态监控器 StateMonitor.h/cpp

## 交易时段状态枚举 SimpleState
```cpp
typedef enum tagSimpleState
{
	SS_ORIGINAL,		// 未初始化状态（0）- 交易日开始前
	SS_INITIALIZED,		// 已初始化状态（1）- 系统就绪等待开盘
	SS_RECEIVING,		// 交易中状态（2）- 正在接收行情数据
	SS_PAUSED,			// 休息中状态（3）- 中途休盘时间
	SS_CLOSED,			// 已收盘状态（4）- 停止接收数据
	SS_PROCING,			// 收盘作业中状态（5）- 正在转储历史数据
	SS_PROCED,			// 盘后已处理状态（6）- 数据已归档
	SS_Holiday	= 99	// 节假日状态（99）- 非交易日
} SimpleState;
```

## 交易时段状态 StateInfo
```cpp
typedef struct _StateInfo
{
	char		_session[16];           // 交易时段标识符（如"TRADING"），最大15字符+'\0'
	uint32_t	_init_time;             // 初始化时间，格式HHMM（如0830表示8:30）
	uint32_t	_close_time;            // 收盘时间，格式HHMM（如1505表示15:05）
	uint32_t	_proc_time;             // 盘后处理时间，格式HHMM（如1530表示15:30）
	SimpleState	_state;                 // 当前状态（状态机的当前状态）
	WTSSessionInfo*	_sInfo;             // 交易时段详细信息指针（包含完整的时段配置）

	typedef struct _Section
	{
		uint32_t _from;                 // 区间开始时间，格式HHMM
		uint32_t _end;                  // 区间结束时间，格式HHMM
	} Section;
	
	std::vector<Section> _sections;     // 交易时间区间集合（支持多个不连续时段）
	
} StateInfo;
```

## 交易时段状态监控器 StateMonitor
用于管理多个交易时段的状态。

### 知识及理解

#### 品种的交易时段模板和节假日模板
总结：
- 任一品种包含`节假日模板`和`交易时段模板`（包含多个`交易时段`）
- 任一`交易时段`属于某个`交易日`，只有其对应的`交易日`不是节假日时，该`交易时段`才属于真正的可交易时间
- `交易时段模板`包含一个偏移
  - `== 0`：其包含的所有`交易时段`的`交易日`就是其所在`日历日`
  - `> 0`：交易时段的开始时间加上该偏移后
    - 小于 24 * 60 = 1440分钟：交易日是其所在日历日
    - 大于等于1440分钟：交易日是其所在日历日的下一天
  - `< 0`：交易日是所在日历日扣除该偏差（时差）后所在的日历日

例如：上海期货交易所（SHFE）的螺纹钢（rb）
- **节假日模板**: 假设2025年的元旦和春节假期安排如下：
  - 元旦: 1月1日（周三）为法定假日，休市。
  - 春节: 1月28日（周二，除夕前一天）晚上起至2月5日（周三）为法定假日，休市。2月6日（周四）恢复交易。
  - 周末: 所有周六、周日休市。
- **交易时段模板**: 螺纹钢的交易时间规则如下，这是一个典型的具有夜盘的品种。
  - 夜盘: 21:00 - 23:00
    - **夜盘时段属于下一个交易日**
    - **系统会为此模板配置一个正的偏移量**，例如 +480 分钟（8小时）
    - **如何判断是否为夜盘？加上偏移量后大于 24 * 60 = 1440 分钟**
  - 日盘 (上午): 09:00 - 10:15, 10:30 - 11:30
  - 日盘 (下午): 13:30 - 15:00
  - **关于偏移量**：
    - == 0；日内
- 结合**节假日模板**和**交易时段模板**推导出的详细交易时间表：
  * **12月31日, 星期二 (2024年)**
    * 日历日: 2024年12月31日
    * 交易日归属:
      * 白天的交易 (09:00 - 15:00) 属于 12月31日交易日。
      * **晚上的夜盘 (21:00 - 23:00) 本应属于下一个交易日。但由于下一个日历日（1月1日）是法定假日，所以今天晚上没有夜盘**。
    * 交易时间: 09:00 - 11:30, 13:30 - 15:00
  * **1月1日, 星期三 (元旦)**
    * 日历日: 2025年1月1日
    * 交易日归属: N/A
    * 交易时间: 全天休市
  * **1月2日, 星期四**
    * 日历日: 2025年1月2日
    * 交易日归属:
      * 白天的交易 (09:00 - 15:00) 属于 1月2日交易日。
      * 晚上的夜盘 (21:00 - 23:00) 属于 1月3日交易日。
    * 交易时间: 09:00 - 11:30, 13:30 - 15:00 以及 21:00 - 23:00
  * **1月3日, 星期五**
    * **日历日**: 2025年1月3日
    * **交易日归属**:
      * 白天的交易 (09:00 - 15:00) 属于 1月3日交易日
      * **晚上：由于下一个日历日（1月4日）是周六（非交易日），所以今天晚上没有夜盘**
    * 交易时间**: **09:00 - 11:30, 13:30 - 15:00
  * **1月4日 (周六) & 1月5日 (周日)**
    * 交易时间: 全天休市
    * 系统判断逻辑: 节假日模板生效，因为是周末

#### 状态变化
场景：郑州商品交易所的 PTA (精对苯二甲酸) 期货。这是一个典型的日盘品种，上午有两节交易，中间有15分钟的小节休息
- 交易时段模板：
  - 9:00-10:15：上午第一节
  - 10:30-11:30：上午第二节
  - 13:30-15:00：下午盘
- 偏移为 0：没有夜盘
- inittime: 08:30 (系统初始化时间)
- closetime: 15:05 (行情接收截止时间，比15:00收盘稍晚以接收最后数据)
- proctime: 15:30 (盘后数据处理开始时间)

一个完整交易日的状态变化：

**交易日：2025年10月10日 (星期五)**

| **时间点** | **当前状态** | **`run()` 方法中的关键判断** | **新状态** | **实际意义与系统行为** |
| :--- | :--- | :--- | :--- | :--- |
| **08:00:00** | `SS_ORIGINAL` | 当前时间 `0800` < `inittime` `0830`。 | `SS_ORIGINAL` | **系统休眠**。`StateMonitor` 处于待机状态，等待初始化时间的到来。此时不接收任何行情数据。 |
| **08:30:00** | `SS_ORIGINAL` | 当前时间 `0830` >= `inittime` `0830`。 | `SS_INITIALIZED` | **系统初始化**。系统被唤醒，进入盘前准备阶段。此时仍不接收行情数据。 |
| **08:59:00** | `SS_INITIALIZED` | 当前时间 `0859` < 第一个交易时段开始时间 `0900`。 | `SS_INITIALIZED` | **等待开盘**。系统已万事俱备，只等开盘信号。 |
| **09:00:00** | `SS_INITIALIZED` | 当前时间 `0900` >= 集合竞价/开盘时间，并且 `isInSections(0900)` 为 `true`。 | `SS_RECEIVING` | **上午第一节开盘**。状态切换到“接收中”。`DataManager` 的数据闸门正式打开，开始接收、存储和广播 `rb` 品种的实时行情。 |
| **10:00:00** | `SS_RECEIVING` | 当前时间 `1000` 仍在 `0900-1015` 区间内。 | `SS_RECEIVING` | **交易进行中**。系统持续接收行情数据。 |
| **10:15:00** | `SS_RECEIVING` | 当前时间 `1015` 不再处于任何交易区间内 (`!isInSections(1015)`)。 | `SS_PAUSED` | **上午小节休盘**。系统进入暂停状态。`DataManager` 停止接收新的行情数据。 |
| **10:25:00** | `SS_PAUSED` | 当前时间 `1025` 仍不处于任何交易区间内。 | `SS_PAUSED` | **小节休息中**。系统保持暂停，等待下一个交易时段的开始。 |
| **10:30:00** | `SS_PAUSED` | 当前时间 `1030` 处于 `1030-1130` 交易区间内 (`isInSections(1030)`)。 | `SS_RECEIVING` | **上午第二节开盘**。状态从“暂停”切换回“接收中”。`DataManager` 再次打开数据闸门。 |
| **11:30:00** | `SS_RECEIVING` | 当前时间 `1130` 不再处于任何交易区间内 (`!isInSections(1130)`)。 | `SS_PAUSED` | **午间休盘**。与上午小节休盘类似，系统再次进入暂停状态，停止接收数据，等待下午开盘。 |
| **12:30:00** | `SS_PAUSED` | 当前时间 `1230` 仍处于午休时段。 | `SS_PAUSED` | **午休进行中**。 |
| **13:30:00** | `SS_PAUSED` | 当前时间 `1330` 处于 `1330-1500` 交易区间内 (`isInSections(1330)`)。 | `SS_RECEIVING` | **下午盘开盘**。状态第三次切换到“接收中”，系统恢复行情接收。 |
| **15:00:00** | `SS_RECEIVING` | 当前时间 `1500` 仍处于 `1330-1500` 区间内（通常区间是左闭右开）。 | `SS_RECEIVING` | **下午收盘**。***但数据接收尚未停止，以确保能收到最后的结算价等收盘数据***。 |
| **15:05:00** | `SS_RECEIVING` | 当前时间 `1505` >= `closetime` `1505`。 | `SS_CLOSED` | **行情接收窗口关闭**。状态切换到“已收盘”。从此以后，`DataManager` 将拒绝所有新的行情数据。 |
| **15:20:00** | `SS_CLOSED` | 当前时间 `1520` < `proctime` `1530`。 | `SS_CLOSED` | **等待盘后处理**。系统处于静默状态，等待预设的数据归档时间。 |
| **15:30:00** | `SS_CLOSED` | 当前时间 `1530` >= `proctime` `1530`。 | `SS_PROCING` | **开始盘后数据处理**。状态切换到“处理中”。`StateMonitor` **立即触发** `_dt_mgr->transHisData(...)`，通知 `DataManager` 开始执行数据转储任务。 |
| **15:30:01** | `SS_PROCING` | 这是一个短暂的过渡状态。 | `SS_PROCED` | **盘后处理完成**。在下一个1秒的检查周期，状态立即切换为“已处理”，表示当天的所有工作已结束。 |
| **23:00:00** | `SS_PROCED` | 当前时间 `2300` > `inittime` `0830`。 | `SS_PROCED` | **当日工作已结束**。系统保持“已处理”状态，等待午夜的到来。 |
| **第二天 01:00:00** | `SS_PROCED` | 当前时间 `0100` 处于 `0000` 和 `inittime` `0830` 之间。 | `SS_ORIGINAL` | **状态重置，迎接新一天**。午夜过后，`StateMonitor` 检测到已进入新的日历日，且时间早于初始化时间，于是将状态重置为 `SS_ORIGINAL`，准备开始下一个完整的交易日生命周期。 |

### 成员

- `StateMap _map`：状态映射表，存储所有交易时段的状态信息
  - 本质上是 **map<交易时段ID, 交易时段状态StateInfo\*\>**
- `WTSBaseDataMgr* _bd_mgr`：基础数据管理器指针
- `DataManager* _dt_mgr`：数据管理器指针
- `StdThreadPtr _thrd`：监控线程智能指针，指向状态监控线程
  - 线程每秒检查一次状态并执行转换。
- `bool _stopped`：停止标志，控制监控线程的运行

### 方法

#### 初始化与生命周期

##### 初始化状态监控器 initialize
```cpp
/* @param filename 状态配置文件路径
 * @param bdMgr 基础数据管理器指针
 * @param dtMgr 数据管理器指针
 * @return bool 初始化成功返回true，失败返回false
 */
bool StateMonitor::initialize(const char* filename, WTSBaseDataMgr* bdMgr, DataManager* dtMgr)
```
配置JSON文件例如：
```json
{
  "TRADING": {			// 交易时段ID
    "inittime": 830,
    "closetime": 1505,
    "proctime": 1530
  },
  "NIGHT": {
    "inittime": 2030,
    "closetime": 2305,
    "proctime": 2330
  }
}
```
流程：
- 将参数 bdMgr 和 dtMgr 设置给基础数据管理器 `_bd_mgr` 和数据管理器 `_dt_mgr`
- 遍历每一个独立的交易时段ID
  - 创建一个新的 stateInfo: *StateInfo，在基础数据管理器 `_bd_mgr` 中查找对应交易时段ID的详细交易时间模板 ssInfo: WTSSessionInfo 
  - 读取并设置 stateInfo 的
    - _sInfo(WTSSessionInfo*)
	- _init_time: 初始化时间，在这个时间点，状态监控器会进入 SS_INITIALIZED 状态
    - _close_time: 收盘时间，这个时间点之后，状态会切换到 SS_CLOSED，停止接收行情数据
    - _proc_time: 盘后处理时间，在这个时间点，系统会开始进行数据转储等盘后作业，状态切换到 SS_PROCING
    - _session
    - 提取集合竞价区间、所有连续竞价区间到 _sections 中
  - 设置 `_map`：_map[stateInfo->_session] = stateInfo
  - 从基础数据管理器 `_bd_mgr` 中获取对应该交易时段ID的所有品种代码
    - 基于 ssInfo 的偏移时间设置 `_bd_mgr` 中这些品种对应节假日模板的当前交易日期（当前日期偏移）

##### 启动状态监控线程 run
创建并启动一个独立的监控线程，线程每秒检查一次时间和状态，并根据预定义的规则进行状态转换。
```cpp
void StateMonitor::run()
```
```mermaid
stateDiagram-v2
    [*] --> SS_ORIGINAL : 系统启动
    SS_ORIGINAL --> SS_INITIALIZED : 到达初始化时间
    SS_INITIALIZED --> SS_RECEIVING : 到达开盘时间
    SS_RECEIVING --> SS_PAUSED : 中途休盘
    SS_PAUSED --> SS_RECEIVING : 恢复交易
    SS_RECEIVING --> SS_CLOSED : 到达收盘时间
    SS_CLOSED --> SS_PROCING : 到达盘后处理时间
    SS_PROCING --> SS_PROCED : 处理完成
    SS_PROCED --> SS_ORIGINAL : 下一交易日
    
    SS_ORIGINAL --> SS_Holiday : 检测到节假日
    SS_INITIALIZED --> SS_Holiday : 检测到节假日
    SS_Holiday --> SS_ORIGINAL : 下一交易日
    
    note right of SS_RECEIVING : 可接收数据状态
    note right of SS_PROCING : 触发历史数据转储
```

流程：监控线程 `_thrd` 未创建时，以如下流程作为线程函数创建线程
- while(!`_stopped`)（持续运行直到触发停止信号）
  - 等待：与上次运行之后流程相距 1 秒
  - 检查停止标志 `_stopped`：触发则结束
  - 获取当前日期 curDate（YYYYMMDD）和时间戳 curMin（HHMM）
  - 遍历 `_map` 中的所有***交易时段***：
    - 获取该交易时段对应的详细配置 sInfo: WTSSessionInfo，并获取 curDate 的对应偏移日期 offDate
    - 根据 sInfo 的状态 `_state`
      - **未初始化 SS_ORIGINAL**
        - 如果：对应该 sInfo 对应的所有品种，当前时间都位于其节假日
          - _state 切换为 SS_Holiday
        - 否则如果：curMin >= sInfo的截止时间（**由于服务器维护、程序崩溃或其他原因，可能在当天的很晚才启动交易程序**）
          - _state 切换为 SS_CLOSED
        - 否则如果：curMin >= sInfo的第一个竞价时段的开始时间（**到达集合竞价时间**）
          - 如果 curMin 在sInfo的某交易区间内：_state 切换为 SS_RECEIVING
          - 否则：
            - 如果 curMin < sInfo的最后一个交易时段的结束时间：_state 切换为 SS_PAUSED
            - 否则：（**也就是大于等于最后一个交易时段的结束时间，并且小于截止时间**）_state 切换为 SS_RECEIVING
        - 否则如果：curMin >= sInfo的初始化时间
          - _state 切换为 SS_INITIALIZED
        - break
      - **已初始化 SS_INITIALIZED**
        - 如果：没有竞价时段或 curMin >= sInfo的第一个竞价时段的开始时间
          - 如果：curMin 不在交易区间内 && curMin < 最后一个交易时段的结束时间
            - _state 切换为 SS_PAUSED
          - 否则：（在交易区间内或大于等于最后一个交易时段的结束时间）
            - _state 切换为 SS_RECEIVING
        - break
      - **接收中 SS_RECEIVING**
        - 如果：curMin >= sInfo 的截止时间
          - _state 切换为 SS_CLOSED
        - 否则如果：curMin >= sInfo的第一个竞价时段的开始时间
          - 如果：curMin < sInfo的最后一个交易时段的结束时间
            - 如果：curMin 不在交易区间内
              - _state 切换为 SS_PAUSED
          - 否则：（大于等于最后一个交易时段的结束时间）
            - 保持
        - break
      - **暂停 SS_PAUSED**
        - 如果：对应该 sInfo 对应的所有品种，当前时间都位于其节假日
          - _state 切换为 SS_Holiday
        - 否则如果：curMin 在交易区间内
          - _state 切换为 SS_RECEIVING
        - break
      - **已收盘 SS_CLOSED**
        - 如果：curMin >= sInfo 的盘后处理时间
          - 如果：该交易时段还未完成盘后处理
            - _state 切换为 SS_PROCING
            - 对 `_dt_mgr` 触发历史数据存储
          - 否则：_state 切换为 SS_PROCED
        - 否则如果：(sInfo的第一个竞价时段的开始时间 <= curMin <= 最后一个交易时段的结束时间) &&  curMin 不在交易区间
          - _state 切换为 SS_PAUSED
        - break
      - **处理中 SS_PROCING**
        - _state 切换为 SS_PROCING（处理中是一个短暂的过渡状态，数据转储完成后立即转换为已处理状态）
        - break
      - **已处理 SS_PROCED**
      - **节假日 SS_Holiday**
        - 如果：(curMin < sInfo的初始化时间) && (对应该 sInfo 对应的所有品种，当前时间至少位于其中一个的交易日（非节假日）)
          - _state 切换为 SS_ORIGINAL
        - break
  - 如果：不是所有的时段都处于节假日状态 && 非节假日的时段都是处理中SS_PROCING状态
    - `_dt_mgr` 触发缓存清理（**所有非节假日的交易时段都已经完成了它们各自的数据接收工作，并都进入了盘后数据处理，这时触发一次全局的、最终的系统资源清理**）

##### 停止状态监控 stop
```cpp
void StateMonitor::stop()
{
	// 设置停止标志
	// 监控线程会在下次循环检查时发现并退出
	_stopped = true;

	// 等待线程结束
	// join()会阻塞当前线程，直到监控线程完全退出
	if (_thrd)
		_thrd->join();
}
```

#### 状态查询接口

##### 检查是否有任一时段处于指定状态 isAnyInState
```cpp
/* @param ss 要检查的状态
    * @return bool 至少有一个时段处于该状态返回true，否则返回false
    */
inline bool	isAnyInState(SimpleState ss) const
{
    auto it = _map.begin();
    for (; it != _map.end(); it++)
    {
        const StatePtr& sInfo = it->second;     // 获取StateInfo智能指针
        if (sInfo->_state == ss)                // 检查状态是否匹配
            return true;                        // 找到匹配的，立即返回true
    }

    return false;
}
```

##### 检查所有非SS_Holiday时段是否都处于指定状态 isAllInState
```cpp
/* @param ss 要检查的状态
 * @return bool 所有非节假日时段都处于该状态返回true，否则返回false
 */
inline bool	isAllInState(SimpleState ss) const
{
    auto it = _map.begin();
    for (; it != _map.end(); it++)
    {
        const StatePtr& sInfo = it->second;     // 获取StateInfo智能指针
        
        // 如果当前时段不是节假日，且状态不等于指定状态
        // 说明不是所有时段都处于指定状态
        if (sInfo->_state != SS_Holiday && sInfo->_state != ss)
            return false;                       // 找到不匹配的，返回false
    }

    return true;
}
```

##### 检查指定时段是否处于指定状态 isInState
```cpp
/* @param sid 交易时段标识符（如"TRADING"）
 * @param ss 要检查的状态
 * @return bool 时段存在且状态匹配返回true，否则返回false
 */
inline bool	isInState(const char* sid, SimpleState ss) const
{
    // 在哈希映射表中查找sid对应的StateInfo
    auto it = _map.find(sid);
    if (it == _map.end())                       // 如果找不到该时段
        return false;                           // 返回false

    // 获取StateInfo智能指针
    const StatePtr& sInfo = it->second;
    
    // 比较状态是否匹配
    return sInfo->_state == ss;
}
```

# 行情解析器适配器 ParserAdapter.h/cpp
```cpp
class ParserAdapter : public IParserSpi, private boost::noncopyable
```
参考 [Includes/note.ipynb/API 接口层/行情解析 IParserApi.h/行情解析器回调接口 IParserSpi](../Includes/note.ipynb)

## 行情解析器适配器类 ParserAdapter

### 成员
- `IParserApi* _parser_api`：Parser实例指针，指向实际的Parser对象（如ParserCTP、ParserXTP等）
  - 参考 [Includes/note.ipynb/API 接口层/行情解析 IParserApi.h/行情解析器接口 IParserApi](../Includes/note.ipynb)
- `FuncDeleteParser _remover`：Parser 销毁函数指针，用于销毁Parser实例
  - typedef void\(\*FuncDeleteParser\)\(wtp::IParserApi* &parser\);
- `WTSBaseDataMgr* _bd_mgr`：基础数据管理器指针
- `DataManager* _dt_mgr`：数据管理器指针，接收行情数据并进行存储和广播
- `IndexFactory* _idx_fact`：指数工厂指针，处理指数计算相关的逻辑
- `bool _stopped`：停止标志，控制适配器是否继续处理数据
- `ExchgFilter _exchg_filter`：交易所过滤器，存储允许订阅的交易所列表
  - typedef wt_hashset\<std::string\> ExchgFilter;
  - 例子：{"SHFE", "DCE"} 表示只允许订阅上期所和大商所
- `ExchgFilter _code_filter`：合约代码过滤器，存储允许订阅的合约代码列表
  - 支持格式："SHFE.rb2105" 即具体合约；或 "SHFE.rb" 即整个品种
  - _code_filter 优先级高于 _exchg_filter
- `WTSVariant* _cfg`：配置对象指针，保存初始化时的配置参数
- `std::string _id`：适配器唯一标识符

### 方法
好的，这是 `ParserAdapter` 类的分层代码段，按照您要求的 Markdown 格式：

- **核心属性与构造函数**
  - 构造函数：`ParserAdapter(WTSBaseDataMgr * bgMgr, DataManager* dtMgr, IndexFactory *idxFactory)`
  - 析构函数：`~ParserAdapter()`
  - 获取适配器ID：`const char* id() const`

- **初始化与生命周期 (Initialization & Lifecycle)**
  - 初始化 (从配置)：`bool init(const char* id, WTSVariant* cfg)`
  - 初始化 (外部注入)：`bool initExt(const char* id, IParserApi* api)`
  - 释放资源：`void release()`
  - 启动运行：`bool run()`

- **IParserSpi 接口回调实现 (IParserSpi Interface Callbacks)**
  - 处理合约列表回调：`virtual void handleSymbolList(const WTSArray* aySymbols) override`
  - 处理Tick行情数据回调：`virtual void handleQuote(WTSTickData *quote, uint32_t procFlag) override`
  - 处理委托队列数据回调：`virtual void handleOrderQueue(WTSOrdQueData* ordQueData) override`
  - 处理逐笔成交数据回调：`virtual void handleTransaction(WTSTransData* transData) override`
  - 处理逐笔委托数据回调：`virtual void handleOrderDetail(WTSOrdDtlData* ordDetailData) override`
  - 处理解析器日志回调：`virtual void handleParserLog(WTSLogLevel ll, const char* message) override`
  - 获取基础数据管理器：`virtual IBaseDataMgr* getBaseDataMgr() override`

#### 初始化与生命周期

##### 初始化 (从配置) init
```cpp
/* @param id 适配器ID
 * @param cfg 配置对象
 * @return bool 初始化成功返回true，失败返回false
 */
bool ParserAdapter::init(const char* id, WTSVariant* cfg)
```
参数 `cfg` 的例子：
```json
{
    "module": "ParserCTP",
    "filter": "SHFE,DCE",
    "code": "SHFE.rb.HOT,CFFEX.IF2503,DCE.i"
}
```
具体流程：
- 保存 id 和 cfg 到成员 `_id` 和 `_cfg` 中
- **（1）动态加载 Parser 模块**
  - 提取配置中的 `module` 字段，并根据 Windows/Linux 平台加上后缀 .dll/.so，查找并加载到 hInst，并从中
    - 获取 createParser 函数并运行，返回的 Paser 实例存储到 `_parser_api: IParserApi*`
    - 获取 deleteParser 函数指针到 `_remover`
- **（2）解析过滤器配置**：提取配置中 `filter`/`code` 字段（以逗号分隔）并保存到 `_exchg_filter`/`_code_filter`
- **（3）初始化 Paser 模块并订阅合约**：
  -  将 this 作为回调接口IParserSpi注册给 `_parser_api: IParserApi*`
  - 如果 `_code_filter` 非空
    - 遍历其所有代码并从 `_bd_mgr: WTSBaseDataMgr*` 中查找
  - 否则如果 `_exchg_filter` 非空
    - 遍历其所有交易所，并从 `_bd_mgr` 找到这些交易所的所有合约
  - 否则
    - 从 `_bd_mgr` 获取所有合约
  - `_parser_api` 订阅所有找到的合约

##### 初始化 (外部提供配置好的Parser实例) initExt
```cpp
/* @param id 适配器ID
 * @param api Parser实例指针
 * @return bool 初始化成功返回true，失败返回false
 */
bool ParserAdapter::initExt(const char* id, IParserApi* api)
```
具体流程：
- 保存 id 和 api 到成员 `_id` 和 `_parser_api: IParserApi*` 中
- 将 this 作为回调接口IParserSpi注册给 `_parser_api`

##### 启动运行 run
就是让 `_parser_api: IParserApi*` 连接服务器。
```cpp
/* @brief 启动Parser连接实现 */
bool ParserAdapter::run()
{
	if (_parser_api == NULL)
		return false;
	// 调用Parser的connect方法，启动连接
	// 连接是异步的，结果通过回调通知
	_parser_api->connect();
	return true;
}
```

#### IParserSpi 接口回调实现

##### 处理合约列表回调 handleSymbolList
```cpp
/* @param aySymbols 合约代码数组 */
void ParserAdapter::handleSymbolList( const WTSArray* aySymbols )
{
	// 当前为空实现：Parser推送的合约列表暂不处理
}
```

##### 处理Tick行情数据回调 handleQuote
```cpp
/**
 * @brief 处理Tick行情数据回调实现（最核心的方法）
 * 
 * 这是ParserAdapter最重要的方法，处理Parser推送的Tick行情数据。
 * 
 * 处理流程：
 * 1. 检查停止标志
 * 2. 验证数据有效性（日期不能为0）
 * 3. 获取或验证合约信息
 * 4. 写入DataManager（存储+广播）
 * 5. 转发给IndexFactory（指数计算）
 * 
 * 数据验证：
 * - actiondate：行情日期，格式YYYYMMDD
 * - tradingdate：交易日期，格式YYYYMMDD
 * - 任一为0表示数据无效，可能是Parser初始化阶段的数据
 * 
 * 合约信息处理：
 * - 优先使用数据对象中的合约信息（Parser可能已设置）
 * - 如果没有，从BaseDataMgr查询
 * - 设置到数据对象，供后续使用
 * 
 * @param quote Tick行情数据指针
 * @param procFlag 处理标志（0=正常，1=仅写入）
 */
void ParserAdapter::handleQuote( WTSTickData *quote, uint32_t procFlag )
{
	// 第一步：检查停止标志
	// 如果适配器已停止，丢弃所有数据
	if (_stopped)
		return;

	// 第二步：验证数据的基本有效性
	// 检查行情日期和交易日期是否有效
	// 日期为0通常表示：
	// 1. Parser初始化阶段的测试数据
	// 2. 数据解析错误
	// 3. 网络传输错误
	if (quote->actiondate() == 0 || quote->tradingdate() == 0)
		return;

	// 第三步：获取合约信息
	// 尝试从Tick对象获取合约信息（Parser可能已设置）
	WTSContractInfo* contract = quote->getContractInfo();
	if (contract == NULL)                       // 如果Tick中没有合约信息
	{
		// 从BaseDataMgr查询合约信息
		contract = _bd_mgr->getContract(quote->code(), quote->exchg());
		
		// 设置合约信息到Tick对象
		// 后续处理（DataManager、IndexFactory）可以直接使用
		// 避免重复查询，提高性能
		quote->setContractInfo(contract);
	}

	// 第四步：验证合约信息
	if (contract == NULL)                       // 如果合约信息不存在
		return;                                 // 丢弃数据（可能是无效合约或配置错误）

	// 第五步：写入DataManager
	// DataManager会：
	// 1. 检查是否可接收（canSessionReceive）
	// 2. 写入磁盘文件
	// 3. 更新内存缓存
	// 4. 广播到所有Caster
	if (!_dt_mgr->writeTick(quote, procFlag))
		return;                                 // 写入失败，不再继续处理

	// 第六步：转发给IndexFactory（指数计算）
	if (_idx_fact)                              // 如果指数工厂存在
		// IndexFactory会检查该Tick是否是指数成分
		// 如果是，触发指数重算
		_idx_fact->handle_quote(quote);
}
```

##### 处理委托队列数据回调 handleOrderQueue

##### 处理逐笔成交数据回调 handleTransaction

##### 处理逐笔委托数据回调 handleOrderDetail

##### 处理解析器日志回调 handleParserLog

##### 获取基础数据管理器 getBaseDataMgr

## 行情解析器适配器管理器类 ParserAdapterMgr